In [ ]:
%pip install scikit-learn scipy seaborn pandas numpy matplotlib

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

os.makedirs("outputs", exist_ok=True)

# Moda de una serie ignorando nulos (se usa en los perfiles)
def moda(s):
    s = s.dropna()
    return s.value_counts().index[0] if len(s) else "NA"

# Parte A: aprendizaje no supervisado
Accidentalidad vial de Medellín (2008-2025). Regla: la variable objetivo (víctimas) **no se usa para agrupar**; se usa al final para validar.

# 1. Calidad de datos

In [ ]:
# Excel en español guarda con ";" como separador y "," como decimal.
# El diccionario de datos dice que "N/D" y "Sin inf" significan dato no disponible: se leen como nulos.
df = pd.read_csv("Fatal_Road_Traffic.csv", sep=";", decimal=",",
                 na_values=["N/D", "Sin inf"], low_memory=False)
print(df.shape)
df.head(3)

In [ ]:
df.info()
df.describe().round(2)

In [ ]:
# Fecha: viene como 01/ene/2008 (mes en español). Se lleva a número con un mapa, como en cars_sucio.
meses = {"ene": "01", "feb": "02", "mar": "03", "abr": "04", "may": "05", "jun": "06",
         "jul": "07", "ago": "08", "sep": "09", "oct": "10", "nov": "11", "dic": "12"}
fecha_txt = df["FECHA_INCIDENTE"].astype(str).str.strip().str.lower()
for es, num in meses.items():
    fecha_txt = fecha_txt.str.replace("/" + es + "/", "/" + num + "/", regex=False)
df["fecha"] = pd.to_datetime(fecha_txt, format="%d/%m/%Y", errors="coerce")
print("Fechas que no se pudieron leer:", df["fecha"].isna().sum())   # debe ser 0

df["hora"] = pd.to_numeric(df["HORA_INCIDENTE"].astype(str).str.split(":").str[0], errors="coerce")   # "00:10:00" -> 0
df["dia_semana"] = df["fecha"].dt.dayofweek            # 0 = lunes ... 6 = domingo
df["fin_de_semana"] = (df["dia_semana"] >= 5).astype(int)
df["anio"] = df["fecha"].dt.year

In [ ]:
# Variable para VALIDAR al final (NO se usa para agrupar): 1 = heridos o muertos, 0 = solo daños
print(df["GRAVEDAD_INCIDENTE"].value_counts())
g = df["GRAVEDAD_INCIDENTE"].astype(str).str.lower()
df["con_victimas"] = np.where(g.str.contains("herid|muert"), 1,
                              np.where(g.str.contains("daño|dano"), 0, np.nan))

df = df.rename(columns={"Latitud": "lat", "Longitud": "lon", "CLASE_INCIDENTE": "clase",
                        "Comuna Planeacion": "comuna"})

# Coordenadas a número (si alguna celda tiene texto, queda como NaN)
for c in ["lat", "lon"]:
    df[c] = pd.to_numeric(df[c].astype(str).str.replace(",", "."), errors="coerce")
print(df[["lat", "lon"]].dtypes)

In [ ]:
# Nulos y valores imposibles (reglas de dominio)
print(df[["lat", "lon", "hora", "con_victimas"]].isna().mean().round(3))   # ~6% sin coordenadas
n0 = len(df)
df = df.dropna(subset=["lat", "lon", "hora", "con_victimas"])
# Según el diccionario, el archivo cae en lat 6.11-6.35 y lon -75.67 a -75.51 (incluye corregimientos rurales).
# Se deja un margen: lo que quede fuera es un error de geocodificación.
df = df[df["lat"].between(6.08, 6.42) & df["lon"].between(-75.72, -75.45)].reset_index(drop=True)
print(f"Filas: {n0:,} -> {len(df):,}")

# Duplicados: LLAVE es única, así que se buscan filas iguales SIN contar LLAVE.
# No se borran (dos incidentes distintos pueden verse iguales); solo se documenta.
cols = [c for c in df.columns if c != "LLAVE"]
print("Filas repetidas (sin contar LLAVE):", df.duplicated(subset=cols).sum())

In [ ]:
# EDA mínimo
print(df["clase"].value_counts(normalize=True).round(3))
df["hora"].value_counts().sort_index().plot.bar()
plt.title("Incidentes por hora"); plt.show()

# 2. Preparación
- **hora y día** son circulares (las 23h están al lado de las 0h): se pasan a seno y coseno.
- **Se agrupa por DÓNDE y CUÁNDO** (lat, lon, hora, día). La **clase del incidente NO entra al clustering**: al probarlo, las dummies 0/1 dominaban la distancia y los grupos resultaron ser simplemente "una clase por grupo". La clase se usa después, en el perfil, para describir cada grupo.
- Sin escalar, lat/lon (decimales) y seno/coseno pesan distinto: StandardScaler.
- Faltantes: se eliminan las filas sin coordenadas (~6 %) porque sin ellas no hay análisis espacial; atípicos geográficos fuera de Medellín se descartan como errores de geocodificación.

In [ ]:
top = df["clase"].value_counts().index[:4]
df["clase_g"] = np.where(df["clase"].isin(top), df["clase"], "Otras")   # solo para el perfil

X = pd.DataFrame({
    "lat": df["lat"], "lon": df["lon"],
    "hora_sin": np.sin(2 * np.pi * df["hora"] / 24), "hora_cos": np.cos(2 * np.pi * df["hora"] / 24),
    "dia_sin": np.sin(2 * np.pi * df["dia_semana"] / 7), "dia_cos": np.cos(2 * np.pi * df["dia_semana"] / 7),
})

scaler = StandardScaler()
Xs = scaler.fit_transform(X)          # en clustering se ajusta con todo el dataset
print(Xs.shape)

# La silueta cuesta O(n²): con cientos de miles de filas se usa sample_size
N_SIL = 10000

# 3. K-means: elegir K con codo + silueta

In [ ]:
ks = range(2, 11)
inercias = [KMeans(k, n_init=10, random_state=42).fit(Xs).inertia_ for k in ks]
siluetas = [silhouette_score(Xs, KMeans(k, n_init=10, random_state=42).fit_predict(Xs),
                             sample_size=N_SIL, random_state=0) for k in ks]

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(ks, inercias, "o-"); ax[0].set(title="Codo", xlabel="K", ylabel="inercia")
ax[1].plot(ks, siluetas, "o-"); ax[1].set(title="Silueta", xlabel="K", ylabel="score")
plt.tight_layout(); plt.savefig("outputs/A1_codo_silueta.png", dpi=150); plt.show()
print(dict(zip(ks, np.round(siluetas, 3))))

In [ ]:
# >>> ELIGE K mirando el codo + la silueta + que los grupos sirvan para decidir. Justifícalo en el informe.
K = 4

kmeans = KMeans(n_clusters=K, n_init=10, random_state=42)
df["km"] = kmeans.fit_predict(Xs)
print(df["km"].value_counts().sort_index())

# Diagrama de silueta (receta del profe) sobre una muestra
idx = np.random.default_rng(0).choice(len(df), N_SIL, replace=False)
s = silhouette_samples(Xs[idx], df["km"].values[idx])
lab = df["km"].values[idx]
y0 = 0
plt.figure(figsize=(6, 4))
for c in np.unique(lab):
    s_c = np.sort(s[lab == c])
    plt.fill_betweenx(np.arange(y0, y0 + len(s_c)), 0, s_c)
    plt.text(-0.05, y0 + len(s_c) / 2, str(c))
    y0 += len(s_c) + 100
plt.axvline(s.mean(), ls="--", color="k"); plt.yticks([])
plt.xlabel("coeficiente de silueta s(i)"); plt.title(f"K-means K={K}")
plt.savefig("outputs/A2_diagrama_silueta.png", dpi=150); plt.show()

# 4. Clustering jerárquico
Necesita todas las distancias entre pares: con más de 650 mil filas no cabe en memoria, así que se usa una **muestra aleatoria de 4000** (decisión a justificar en el informe).

In [ ]:
muestra = df.sample(4000, random_state=42)
Xm = Xs[muestra.index]

Z = linkage(Xm, method="ward")                       # 1. construir el árbol
plt.figure(figsize=(12, 4))
dendrogram(Z, truncate_mode="lastp", p=30, no_labels=True)   # 2. dibujarlo (solo los últimos 30 cortes)
plt.title("Dendrograma (Ward, muestra de 4000)")
plt.savefig("outputs/A3_dendrograma.png", dpi=150); plt.show()

muestra = muestra.assign(jer=fcluster(Z, t=K, criterion="maxclust"))   # 3. cortar en K grupos
muestra["jer"].value_counts()

In [ ]:
# Comparar linkages (single encadena; ward/complete dan grupos más balanceados)
for met in ["single", "complete", "average", "ward"]:
    print(met, np.bincount(fcluster(linkage(Xm, method=met), t=K, criterion="maxclust"))[1:])

# Jerárquico vs K-means sobre la misma muestra: tabla cruzada (las etiquetas son arbitrarias)
pd.crosstab(muestra["km"], muestra["jer"])

# 5. DBSCAN (haversine): puntos calientes en el mapa
Igual que el caso California de la clase:
- lat/lon NO se escalan (ya comparten unidad): métrica haversine, coordenadas en radianes, eps = km / 6371 para que se lea en kilómetros.
- Solo lugar: un hotspot es un sitio. La hora y la clase se miran después, en el perfil.
- Truco por tamaño: muchos incidentes ocurren en la misma esquina. Se agrupan por coordenada redondeada (~11 m) y cada punto único pesa lo que repite (`sample_weight`). Es el mismo DBSCAN, solo más rápido.

In [ ]:
R = 6371   # radio de la Tierra (km)
df["lat_r"] = df["lat"].round(4)
df["lon_r"] = df["lon"].round(4)
pts = df.groupby(["lat_r", "lon_r"]).size().rename("w").reset_index()
coords = np.radians(pts[["lat_r", "lon_r"]])
print(f"{len(pts):,} puntos únicos de {len(df):,} incidentes")

In [ ]:
# Gráfico k-distancia (receta del profe, con haversine; resultado en metros)
min_samples = 30
nn = NearestNeighbors(n_neighbors=min_samples, metric="haversine", algorithm="ball_tree").fit(coords)
dist, _ = nn.kneighbors(coords)
k_dist = np.sort(dist[:, -1]) * R * 1000
plt.plot(k_dist); plt.yscale("log")
plt.xlabel("puntos ordenados"); plt.ylabel(f"distancia al vecino {min_samples} (m)")
plt.savefig("outputs/A4_k_distancia.png", dpi=150); plt.show()

In [ ]:
def correr_dbscan(eps_m, min_samples):
    db = DBSCAN(eps=(eps_m / 1000) / R, min_samples=min_samples,
                metric="haversine", algorithm="ball_tree")
    return db.fit_predict(coords, sample_weight=pts["w"])

# Malla: se reporta SIEMPRE el % de ruido y el % de incidentes en el grupo más grande.
# Si un grupo tiene casi todo, DBSCAN conectó toda la ciudad (efecto cadena): esa configuración no sirve.
filas = []
for eps_m in [30, 50, 75, 100, 150]:
    for ms in [100, 300, 600, 1000]:
        lab = correr_dbscan(eps_m, ms)
        pesos = pts["w"].groupby(lab).sum()          # incidentes por grupo
        ruido = pesos.get(-1, 0) / pesos.sum()
        validos = pesos.drop(-1, errors="ignore")
        filas.append((eps_m, ms, len(validos), round(ruido, 3),
                      round(validos.max() / pesos.sum(), 3) if len(validos) else 0))
malla = pd.DataFrame(filas, columns=["eps_m", "min_samples", "n_hotspots", "pct_ruido", "pct_en_el_mayor"])
malla

In [ ]:
# >>> ELIGE de la malla de arriba: pct_en_el_mayor bajo (< 0.3), ruido entre 10% y 60%,
# y un número de hotspots que se pueda interpretar. Justifícalo en el informe.
EPS_M, MIN_SAMPLES = 75, 300

pts["db"] = correr_dbscan(EPS_M, MIN_SAMPLES)
df = df.merge(pts[["lat_r", "lon_r", "db"]], on=["lat_r", "lon_r"])

n_hot = df.loc[df["db"] != -1, "db"].nunique()
pct_ruido = (df["db"] == -1).mean()
pct_mayor = df["db"].value_counts().drop(-1, errors="ignore").max() / len(df) if n_hot else 0
print(f"{n_hot} hotspots · {pct_ruido:.0%} ruido · el grupo más grande tiene {pct_mayor:.0%} de los incidentes")
if pct_mayor > 0.3:
    print("ATENCIÓN: un solo grupo domina. Baja eps o sube min_samples.")

# Silueta sin ruido, en haversine, con sample_size (como en California). Se reporta junto al % de ruido.
mask = (df["db"] != -1).to_numpy()
c_all = np.radians(df[["lat", "lon"]].to_numpy())
if n_hot >= 2:
    sil_db = silhouette_score(c_all[mask], df["db"].to_numpy()[mask], metric="haversine",
                              sample_size=N_SIL, random_state=0)
else:
    sil_db = np.nan
print("silueta DBSCAN (sin ruido):", round(sil_db, 3))

In [ ]:
# Mapa: DBSCAN vs K-means
fig, ax = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)
r = df["db"] == -1
ax[0].scatter(df.loc[r, "lon"], df.loc[r, "lat"], s=1, c="lightgray")
ax[0].scatter(df.loc[~r, "lon"], df.loc[~r, "lat"], s=1, c=df.loc[~r, "db"] % 20, cmap="tab20")
ax[0].set_title(f"DBSCAN: {n_hot} hotspots, {r.mean():.0%} ruido (gris)")
ax[1].scatter(df["lon"], df["lat"], s=1, c=df["km"], cmap="tab10")
ax[1].set_title(f"K-means K={K}")
plt.tight_layout(); plt.savefig("outputs/A5_mapas.png", dpi=150); plt.show()

# 6. Comparación
La silueta **no** es comparable entre métodos con geometrías distintas (K-means y jerárquico agrupan por lugar+hora+día; DBSCAN solo por lugar): hay que leerla junto con las observaciones.

In [ ]:
comparacion = pd.DataFrame({
    "método": ["K-means", "Jerárquico (Ward)", "DBSCAN haversine"],
    "parámetros": [f"K={K}", f"corte K={K}", f"eps={EPS_M} m, min_samples={MIN_SAMPLES}"],
    "espacio": ["lugar+hora+día"] * 2 + ["solo lugar"],
    "grupos": [K, K, n_hot],
    "% ruido": [0, 0, round(pct_ruido * 100, 1)],
    "silueta": [round(silhouette_score(Xs, df["km"], sample_size=N_SIL, random_state=0), 3),
                round(silhouette_score(Xm, muestra["jer"]), 3), round(sil_db, 3)],
})
comparacion

# 7. Interpretación: perfiles y validación externa
Recién aquí se mira la gravedad. Si grupos formados **sin** ella difieren en la tasa de víctimas, la estructura es real y sirve para decidir.

In [ ]:
dummies = pd.get_dummies(df["clase_g"], prefix="clase", dtype=float)
base = pd.concat([df[["hora", "fin_de_semana", "con_victimas"]], dummies], axis=1)

perfil = base.groupby(df["km"]).mean().round(2)
perfil.insert(0, "n", df["km"].value_counts().sort_index())
perfil.insert(1, "comuna_top", df.groupby("km")["comuna"].agg(moda))
perfil

In [ ]:
# Heatmap en z-score (como el de USArrests): ¿qué es alto o bajo en cada grupo respecto al promedio?
num = perfil.drop(columns=["n", "comuna_top"])
sns.heatmap((num - num.mean()) / num.std(), annot=num, cmap="RdBu_r", center=0)
plt.title("Perfil de cada grupo K-means"); plt.savefig("outputs/A6_perfiles.png", dpi=150); plt.show()

In [ ]:
# Validación externa: χ² de independencia entre grupo y víctimas
tabla = pd.crosstab(df["km"], df["con_victimas"])
chi2, p, gl, esperadas = stats.chi2_contingency(tabla)
print(f"Tasa global de víctimas: {df['con_victimas'].mean():.3f} | χ² p-valor = {p:.3g}")
print("Frecuencia esperada mínima:", esperadas.min().round(1))
print("OJO: con n tan grande casi todo sale 'significativo'; mira la DIFERENCIA en tasa, no solo el p-valor.")

In [ ]:
# DBSCAN: los 10 hotspots más grandes vs ruido (los incidentes aislados NO son basura: son información)
grandes = df.loc[df["db"] != -1, "db"].value_counts().head(10).index
df["zona"] = np.where(df["db"].isin(grandes), df["db"], np.where(df["db"] == -1, -1, -2))
df[df["zona"] != -2].groupby("zona").agg(
    n=("hora", "size"), hora_media=("hora", "mean"), fin_semana=("fin_de_semana", "mean"),
    clase_top=("clase_g", moda), comuna_top=("comuna", moda),
    tasa_victimas=("con_victimas", "mean")).round(2)

In [ ]:
# Cruce de métodos: patrones que persisten = estructura real
pd.crosstab(df["km"], np.where(df["db"] == -1, "aislado", "en hotspot"), normalize="index").round(2)

# 8. Perfilado
Para la solución más útil (K-means: cuándo y qué tipo de incidente; DBSCAN: dónde): medias o medianas, gráficos, **nombre de negocio** y **acción concreta** para el decisor.

In [ ]:
# Medianas y medias de las variables que caracterizan cada grupo
df["noche"] = ((df["hora"] >= 22) | (df["hora"] <= 5)).astype(int)
df["hora_pico"] = df["hora"].isin([6, 7, 8, 17, 18, 19]).astype(int)
resumen = df.groupby("km").agg(
    n=("hora", "size"),
    hora_mediana=("hora", "median"),
    pct_noche=("noche", "mean"),
    pct_hora_pico=("hora_pico", "mean"),
    pct_fin_semana=("fin_de_semana", "mean"),
    tasa_victimas=("con_victimas", "mean"),
).round(2)
resumen["pct_del_total"] = (resumen["n"] / len(df)).round(3)
resumen["clase_top"] = df.groupby("km")["clase_g"].agg(moda)
resumen["comuna_top"] = df.groupby("km")["comuna"].agg(moda)
resumen

In [ ]:
# Gráficos del perfil: hora por grupo, mezcla de clases y tasa de víctimas
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
sns.boxplot(data=df.sample(50000, random_state=1), x="km", y="hora", ax=ax[0])
ax[0].set(title="Hora del incidente por grupo", xlabel="grupo")
pd.crosstab(df["km"], df["clase_g"], normalize="index").plot.bar(stacked=True, ax=ax[1])
ax[1].set(title="Clase de incidente por grupo", xlabel="grupo", ylabel="proporción")
ax[2].bar(resumen.index.astype(str), resumen["tasa_victimas"])
ax[2].axhline(df["con_victimas"].mean(), color="r", ls="--", label="global")
ax[2].set(title="Tasa con víctimas (validación externa)", xlabel="grupo"); ax[2].legend()
plt.tight_layout(); plt.savefig("outputs/A7_perfil_grupos.png", dpi=150); plt.show()

In [ ]:
# >>> TÚ llenas nombre y acción mirando la tabla `resumen` y los gráficos de arriba.
# Reglas de la clase: el nombre sale del perfil (p. ej. "Sur, alta violencia"), no del número del grupo.
# La acción debe ser concreta: qué se hace, dónde y cuándo.
nombres = {k: ("COMPLETAR nombre", "COMPLETAR acción") for k in resumen.index}
# Ejemplo de cómo llenarlo (borra o cambia):
# nombres[0] = ("Hora pico en el centro", "Reforzar agentes de tránsito 6-9h y 17-19h en las comunas del grupo")

perfilado = resumen[["n", "pct_del_total", "hora_mediana", "pct_noche", "tasa_victimas", "clase_top", "comuna_top"]].copy()
perfilado["nombre"] = [nombres[k][0] for k in perfilado.index]
perfilado["accion"] = [nombres[k][1] for k in perfilado.index]
perfilado.to_csv("outputs/A8_perfilado.csv")
perfilado

# 9. Puente a la Parte B: DBSCAN + KNN
DBSCAN no tiene `predict()`. Para asignar hotspot a incidentes **nuevos** (el test 2023-2025) se entrena un KNN solo con los puntos núcleo del train, y lo que queda lejos de todo núcleo es "aislado" (-1). Así el hotspot se ajusta solo con train y no hay fuga de datos.

In [ ]:
train = df[df["anio"] <= 2022]
test = df[df["anio"] > 2022]

pt = train.groupby(["lat_r", "lon_r"]).size().rename("w").reset_index()
ct = np.radians(pt[["lat_r", "lon_r"]])
dbt = DBSCAN(eps=(EPS_M / 1000) / R, min_samples=MIN_SAMPLES, metric="haversine", algorithm="ball_tree")
lab_t = dbt.fit_predict(ct, sample_weight=pt["w"])

nucleos = ct.to_numpy()[dbt.core_sample_indices_]
et_nucleos = lab_t[dbt.core_sample_indices_]
knn = KNeighborsClassifier(n_neighbors=1, metric="haversine").fit(nucleos, et_nucleos) if len(nucleos) else None

def asignar_hotspot(d):
    c = np.radians(d[["lat", "lon"]].to_numpy())
    dist, _ = knn.kneighbors(c, n_neighbors=1)
    y = knn.predict(c)
    y[dist[:, 0] > (EPS_M / 1000) / R] = -1       # lejos de todo núcleo = aislado
    return y

if len(test) and len(nucleos):
    print("% aislados en test:", (asignar_hotspot(test) == -1).mean().round(3))
else:
    print("Sin puntos núcleo en train: ajusta EPS_M y MIN_SAMPLES.")

df[["lat", "lon", "hora", "dia_semana", "anio", "clase", "comuna", "con_victimas", "km", "db"]] \
    .to_csv("outputs/parte_A_etiquetas.csv", index=False)